# Prefix Recovery & Synthetic Error Injection: Summary

**Dataset**: IMOBench (343 competition math problems)  
**Models**: Qwen3-4B-Instruct-2507, Qwen3-30B-A3B-Instruct-2507 (local), Gemini 3 Flash, DeepSeek V3.2, MiniMax M2.5 (API)  
**Date**: April 1, 2026

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

FIGDIR = '../figures'

C_BLUE = '#2563eb'
C_ORANGE = '#ea580c'
C_GREEN = '#16a34a'
C_RED = '#dc2626'
C_PURPLE = '#7c3aed'
C_GRAY = '#6b7280'
C_TEAL = '#0d9488'

## 1. Baseline Accuracy

| Model | Correct | Total | Accuracy |
|-------|---------|-------|----------|
| Qwen3-4B-Instruct-2507 | 125 | 400 | 31.2% |
| Qwen3-30B-A3B-Instruct-2507 | 145 | 400 | 36.2% |
| Gemini 3 Flash | 158 | 400 | 39.5% |
| DeepSeek V3.2 | 179 | 400 | 44.8% |
| MiniMax M2.5 | 75 | 400 | 18.8% |

## 2. Direct Recovery vs 2-Turn Revision

| Model | Wrong Baselines | Direct Recovery | 2-Turn Recovery |
|-------|----------------|-----------------|------------------|
| Qwen3-4B | 263 | 0/263 (0.0%) | 6/263 (2.3%) |
| Qwen3-30B-A3B | 228 | 1/228 (0.4%) | -- |
| Gemini 3 Flash | 227 | 1/227 (0.4%) | 20/286 (7.0%) |
| DeepSeek V3.2 | 201 | 9/201 (4.5%) | 21/201 (10.4%) |
| MiniMax M2.5 | 336 | 15/336 (4.5%) | 53/310 (17.1%) |

In [ ]:
# Figure 1: Direct Recovery vs 2-Turn Revision

models = ['Qwen3-4B', 'Qwen3-30B-A3B', 'Gemini 3 Flash', 'DeepSeek V3.2', 'MiniMax M2.5']
direct   = [0.0, 0.4, 0.4, 4.5, 4.5]
revision = [2.3, None, 7.0, 10.4, 17.1]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(models))
w = 0.35

bars1 = ax.bar(x - w/2, direct, w, label='Direct prefix continuation', color=C_BLUE, edgecolor='white', linewidth=0.5)
rev_vals = [v if v is not None else 0 for v in revision]
rev_colors = [C_ORANGE if v is not None else 'none' for v in revision]
bars2 = ax.bar(x + w/2, rev_vals, w, label='2-turn revision', color=rev_colors, edgecolor='white', linewidth=0.5)

for bar, v in zip(bars1, direct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{v:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar, v in zip(bars2, revision):
    if v is not None:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    else:
        ax.text(bar.get_x() + bar.get_width()/2, 0.3,
                'N/A', ha='center', va='bottom', fontsize=8, color=C_GRAY)

ax.set_ylabel('Recovery Rate (%)', fontsize=12)
ax.set_title('Self-Correction from Own Incorrect Solutions (IMOBench)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=10)
ax.legend(fontsize=10, loc='upper left')
ax.set_ylim(0, 22)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(f'{FIGDIR}/recovery_vs_revision.png', dpi=200, bbox_inches='tight')
plt.show()

## 3. Prefix Length Ablation (Incorrect Prefix)

### Qwen3-4B-Instruct-2507 (4,208 samples/length)

| Prefix | Correct | Accuracy |
|--------|---------|----------|
| 0 | 400 | 9.5% |
| 50 | 384 | 9.1% |
| 100 | 370 | 8.8% |
| 200 | 387 | 9.2% |
| 400 | 361 | 8.6% |
| 800 | 362 | 8.6% |
| 1000 | 359 | 8.5% |

### Qwen3-30B-A3B-Instruct-2507 (3,648 samples/length)

| Prefix | Correct | Accuracy |
|--------|---------|----------|
| 0 | 497 | 13.6% |
| 50 | 455 | 12.5% |
| 100 | 475 | 13.0% |
| 200 | 458 | 12.6% |
| 400 | 476 | 13.0% |
| 800 | 470 | 12.9% |
| 1000 | 410 | 11.2% |
| 2000 | 361 | 9.9% |
| 3000 | 355 | 9.7% |
| 4000 | 292 | 8.0% |

In [ ]:
# Figure 2: Prefix Length Ablation

lengths_4b  = [0, 50, 100, 200, 400, 800, 1000]
acc_4b      = [9.5, 9.1, 8.8, 9.2, 8.6, 8.6, 8.5]

lengths_30b = [0, 50, 100, 200, 400, 800, 1000, 2000, 3000, 4000]
acc_30b     = [13.6, 12.5, 13.0, 12.6, 13.0, 12.9, 11.2, 9.9, 9.7, 8.0]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(lengths_4b, acc_4b, 'o-', color=C_BLUE, linewidth=2, markersize=7, label='Qwen3-4B-Instruct-2507')
ax.plot(lengths_30b, acc_30b, 's-', color=C_ORANGE, linewidth=2, markersize=7, label='Qwen3-30B-A3B-Instruct-2507')

ax.axhline(9.5, color=C_BLUE, linestyle='--', alpha=0.3, linewidth=1)
ax.axhline(13.6, color=C_ORANGE, linestyle='--', alpha=0.3, linewidth=1)

ax.set_xlabel('Incorrect Prefix Length (tokens)', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Continuation Accuracy After Incorrect Prefix (IMOBench, 16 samples/problem)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(5, 18)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(f'{FIGDIR}/prefix_length_ablation.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Synthetic Error Injection: Clean vs Corrupted Correct Prefixes

### Clean Correct-Prefix Baselines

| Model | p=200 | p=400 | p=800 |
|-------|-------|-------|-------|
| Qwen3-4B | 1763/2192 (80.4%) | 1756/2176 (80.7%) | 1776/2192 (81.0%) |
| Qwen3-30B | 2354/2752 (85.5%) | 2350/2752 (85.4%) | 2373/2752 (86.2%) |

### Result Perturbation

| Condition | 4B p200 | 4B p400 | 4B p800 | 30B p200 | 30B p400 | 30B p800 |
|-----------|---------|---------|---------|----------|----------|----------|
| Clean | 80.4% | 80.7% | 81.0% | 85.5% | 85.4% | 86.2% |
| c=1 | 79.8% (1750/2192) | 79.8% (1749/2192) | 80.7% (1770/2192) | 85.5% (2354/2752) | 83.7% (2304/2752) | 85.8% (2361/2752) |
| c=3 | 79.7% (1746/2192) | 80.2% (1759/2192) | 80.3% (1761/2192) | 82.1% (2259/2752) | 84.8% (2334/2752) | 85.4% (2351/2752) |
| c=5 | 79.2% (1735/2192) | 78.5% (1721/2192) | 78.9% (1730/2192) | 82.4% (2269/2752) | 83.0% (2284/2752) | 84.9% (2337/2752) |

### Number Swap

| Condition | 4B p200 | 4B p400 | 4B p800 | 30B p200 | 30B p400 | 30B p800 |
|-----------|---------|---------|---------|----------|----------|----------|
| Clean | 80.4% | 80.7% | 81.0% | 85.5% | 85.4% | 86.2% |
| c=1 | 79.9% (1752/2192) | 79.4% (1741/2192) | 79.7% (1748/2192) | 84.7% (2331/2752) | 85.2% (2345/2752) | 85.5% (2352/2752) |
| c=3 | 80.0% (1754/2192) | 78.1% (1713/2192) | 79.0% (1731/2192) | 84.7% (2332/2752) | 84.6% (2327/2752) | 84.3% (2321/2752) |
| c=5 | 79.6% (1744/2192) | 79.3% (1739/2192) | 78.2% (1714/2192) | 84.6% (2327/2752) | 84.4% (2324/2752) | 85.0% (2338/2752) |

In [ ]:
# Figure 3: Corruption Effect — Grouped Bar Chart

data_4b = {
    'clean':   {200: 80.4, 400: 80.7, 800: 81.0},
    ('rp', 1): {200: 79.8, 400: 79.8, 800: 80.7},
    ('rp', 3): {200: 79.7, 400: 80.2, 800: 80.3},
    ('rp', 5): {200: 79.2, 400: 78.5, 800: 78.9},
    ('ns', 1): {200: 79.9, 400: 79.4, 800: 79.7},
    ('ns', 3): {200: 80.0, 400: 78.1, 800: 79.0},
    ('ns', 5): {200: 79.6, 400: 79.3, 800: 78.2},
}
data_30b = {
    'clean':   {200: 85.5, 400: 85.4, 800: 86.2},
    ('rp', 1): {200: 85.5, 400: 83.7, 800: 85.8},
    ('rp', 3): {200: 82.1, 400: 84.8, 800: 85.4},
    ('rp', 5): {200: 82.4, 400: 83.0, 800: 84.9},
    ('ns', 1): {200: 84.7, 400: 85.2, 800: 85.5},
    ('ns', 3): {200: 84.7, 400: 84.6, 800: 84.3},
    ('ns', 5): {200: 84.6, 400: 84.4, 800: 85.0},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, data, title in [(axes[0], data_4b, 'Qwen3-4B-Instruct-2507'),
                         (axes[1], data_30b, 'Qwen3-30B-A3B-Instruct-2507')]:
    prefixes = [200, 400, 800]
    conditions = ['Clean', 'RP c=1', 'RP c=3', 'RP c=5', 'NS c=1', 'NS c=3', 'NS c=5']
    keys = ['clean', ('rp',1), ('rp',3), ('rp',5), ('ns',1), ('ns',3), ('ns',5)]
    colors = [C_GREEN, '#93c5fd', C_BLUE, '#1e3a5f', '#fdba74', C_ORANGE, '#9a3412']

    x = np.arange(len(prefixes))
    n = len(conditions)
    w = 0.11
    offsets = np.linspace(-(n-1)*w/2, (n-1)*w/2, n)

    for i, (key, cond, color) in enumerate(zip(keys, conditions, colors)):
        vals = [data[key][p] for p in prefixes]
        ax.bar(x + offsets[i], vals, w, label=cond, color=color, edgecolor='white', linewidth=0.3)

    ax.set_xticks(x)
    ax.set_xticklabels([f'p={p}' for p in prefixes], fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)

axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_ylim(75, 90)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend(fontsize=8, loc='lower left', ncol=2)

fig.suptitle('Correct-Prefix Continuation: Clean vs. Corrupted (IMOBench, 16 samples)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(f'{FIGDIR}/corruption_grouped_bars.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 4: Dose-Response Curves

rp_4b = {
    200: [80.4, 79.8, 79.7, 79.2],
    400: [80.7, 79.8, 80.2, 78.5],
    800: [81.0, 80.7, 80.3, 78.9],
}
ns_4b = {
    200: [80.4, 79.9, 80.0, 79.6],
    400: [80.7, 79.4, 78.1, 79.3],
    800: [81.0, 79.7, 79.0, 78.2],
}
rp_30b = {
    200: [85.5, 85.5, 82.1, 82.4],
    400: [85.4, 83.7, 84.8, 83.0],
    800: [86.2, 85.8, 85.4, 84.9],
}
ns_30b = {
    200: [85.5, 84.7, 84.7, 84.6],
    400: [85.4, 85.2, 84.6, 84.4],
    800: [86.2, 85.5, 84.3, 85.0],
}

x_vals = [0, 1, 3, 5]
colors_p = {200: C_BLUE, 400: C_ORANGE, 800: C_GREEN}
markers = {200: 'o', 400: 's', 800: '^'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)

for ax, rp, ns, title in [(axes[0], rp_4b, ns_4b, 'Qwen3-4B-Instruct-2507'),
                            (axes[1], rp_30b, ns_30b, 'Qwen3-30B-A3B-Instruct-2507')]:
    for p in [200, 400, 800]:
        ax.plot(x_vals, rp[p], f'{markers[p]}-', color=colors_p[p], linewidth=2,
                markersize=8, label=f'p={p} (result perturb.)')
        ax.plot(x_vals, ns[p], f'{markers[p]}--', color=colors_p[p], linewidth=1.5,
                markersize=6, alpha=0.6, label=f'p={p} (number swap)')

    ax.set_xlabel('Number of Corruptions', fontsize=12)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticks(x_vals)
    ax.set_xticklabels(['0\n(clean)', '1', '3', '5'])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)

axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_ylim(76, 90)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend(fontsize=7.5, loc='lower left', ncol=2)

fig.suptitle('Dose-Response: Accuracy vs. Number of Arithmetic Corruptions', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(f'{FIGDIR}/corruption_dose_response.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 5: Accuracy Drop Heatmap

conditions = ['RP c=1', 'RP c=3', 'RP c=5', 'NS c=1', 'NS c=3', 'NS c=5']
col_labels = ['4B p200', '4B p400', '4B p800', '30B p200', '30B p400', '30B p800']

clean_4b = [80.4, 80.7, 81.0]
clean_30b = [85.5, 85.4, 86.2]
clean_all = clean_4b + clean_30b

corrupted = {
    'RP c=1': [79.8, 79.8, 80.7, 85.5, 83.7, 85.8],
    'RP c=3': [79.7, 80.2, 80.3, 82.1, 84.8, 85.4],
    'RP c=5': [79.2, 78.5, 78.9, 82.4, 83.0, 84.9],
    'NS c=1': [79.9, 79.4, 79.7, 84.7, 85.2, 85.5],
    'NS c=3': [80.0, 78.1, 79.0, 84.7, 84.6, 84.3],
    'NS c=5': [79.6, 79.3, 78.2, 84.6, 84.4, 85.0],
}

drops = np.array([[clean_all[j] - corrupted[c][j] for j in range(6)] for c in conditions])

fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(drops, cmap='YlOrRd', aspect='auto', vmin=0, vmax=4)

for i in range(len(conditions)):
    for j in range(6):
        val = drops[i, j]
        color = 'white' if val > 2.5 else 'black'
        ax.text(j, i, f'{val:+.1f}', ha='center', va='center', fontsize=10, fontweight='bold', color=color)

ax.set_xticks(range(6))
ax.set_xticklabels(col_labels, fontsize=10)
ax.set_yticks(range(len(conditions)))
ax.set_yticklabels(conditions, fontsize=10)
ax.set_title('Accuracy Drop from Clean Correct Prefix (pp)', fontsize=13, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Accuracy Drop (pp)', fontsize=10)

plt.tight_layout()
fig.savefig(f'{FIGDIR}/corruption_drop_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

## Key Takeaways

1. **Self-correction is near-zero**: Direct recovery 0.0-4.5%; 2-turn revision reaches 17.1% at best. Models are strongly anchored to wrong reasoning paths.

2. **Short wrong prefixes don't hurt, long ones do**: Accuracy is flat at ~9% (4B) / ~13% (30B) for prefixes up to 1000 tokens, but drops to 8.0% at 4000 tokens for 30B.

3. **Arithmetic corruption causes only small degradation (0.3-3.4 pp)**: Even 5 corrupted numbers barely dent accuracy. Models route around injected errors rather than propagating them — suggesting CoT functions as context/priming, not active computation.

4. **No clear dose-response**: Corruption count vs. accuracy loss is noisy, not monotonic.

5. **Practical implication**: Iterative refinement without external verification feedback is unlikely to help — models can't reliably self-correct.